In [ ]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import glob
import random
import yaml
import ultralytics


# --- CONFIGURACIÓN DE RUTAS ---
ruta_yaml = '../data/dataset_mini/data.yaml'
nombre_experimento = 'yolov11n'
directorio_salida = '../notebooks/runs/models'

In [ ]:
# --- ENTRENAMIENTO ---
model = YOLO('yolo11n.pt')

results = model.train(
    data=ruta_yaml,

    # Configuración de guardado
    project=directorio_salida,
    name=nombre_experimento,
    exist_ok=True,

    # Hiperparámetros
    epochs=200,
    imgsz=640,
    batch=16,
# --- Hardware ---
    device=[-1, -1],
    workers=8,

    patience=15,
    save=True,

    cls_pw = 1.0,

    # --- Ajustes Finos de Aprendizaje ---
    optimizer='auto',
    cos_lr=True,

    # Gráficas y visualización
    plots=True,
    val=True
)

In [ ]:
model = YOLO('/home/jainogue/yolo/notebooks/runs/models/yolov8s/weights/last.pt')

# 2. Lanzas el entrenamiento con resume=True
results = model.train(resume=True)

In [ ]:
# --- 3. VISUALIZAR MATRIZ DE CONFUSIÓN ---
path_confusion = os.path.join(directorio_salida, nombre_experimento, 'confusion_matrix.png')

if os.path.exists(path_confusion):
    img_cm = cv2.imread(path_confusion)
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(img_cm, cv2.COLOR_BGR2RGB))
    plt.title("Matriz de Confusión Final")
    plt.axis('off')
    plt.show()
else:
    print("La matriz de confusión aún no se ha generado (o hubo un error en validación).")

In [ ]:
# --- 4. COMPARATIVA VISUAL: GROUND TRUTH vs PREDICCIÓN ---
%matplotlib inline
def dibujar_cajas_reales(img_path, label_path, class_names):
    """Dibuja las cajas del archivo .txt sobre la imagen para ver la 'Realidad'"""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])

            # Des-normalizar coordenadas
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)

            # Dibujar caja verde (Ground Truth)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            name = class_names[cls] if cls < len(class_names) else str(cls)
            cv2.putText(img, name, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return img

def comparar_resultados(base_path, model, num_samples=5):
    val_images_path = os.path.join(base_path, 'valid', 'images')
    images = glob.glob(os.path.join(val_images_path, '*.*'))

    if len(images) < num_samples:
        print("No hay suficientes imágenes en valid.")
        return

    samples = random.sample(images, num_samples)
    class_names = model.names # Obtener nombres de clases del modelo entrenado

    for i, img_path in enumerate(samples):
        # 1. Preparar Ground Truth (Izquierda)
        label_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'
        img_gt = dibujar_cajas_reales(img_path, label_path, class_names)

        # 2. Preparar Predicción (Derecha)
        results = model.predict(img_path, conf=0.25, verbose=False)
        res_plotted = results[0].plot() # Esto devuelve array BGR
        img_pred = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

        # 3. Mostrar lado a lado
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        axes[0].imshow(img_gt)
        axes[0].set_title("Original (Etiqueta Manual)")
        axes[0].axis('off')

        axes[1].imshow(img_pred)
        axes[1].set_title("Predicción del Modelo")
        axes[1].axis('off')

        plt.suptitle(f"Ejemplo {i+1}: {os.path.basename(img_path)}")
        plt.tight_layout()
        plt.show()

model = YOLO('/home/jainogue/yolo/notebooks/runs/models/yolov11n/weights/best.pt')
comparar_resultados('../data/dataset', model, num_samples=5)

In [ ]:
from ultralytics import YOLO

model_paths = [
    "runs/models/yolov8n/weights/best.pt",
    "runs/models/yolov8s/weights/best.pt",
    "runs/models/yolov11n/weights/best.pt",
    "runs/models/yolov11s/weights/best.pt"
]

for path in model_paths:
    print(f"\n--- Procesando: {path} ---")
    try:
        model = YOLO(path)
        onnx_path = model.export(
            format="onnx",
            imgsz=640,
            dynamic=False,
            opset=11,
            simplify=True
        )
        print(f"Éxito: Guardado en {onnx_path}")
    except Exception as e:
        print(f"Error procesando {path}: {e}")